![Los Angeles skyline](la_skyline.jpg)

Los Ángeles, California 😎. La Ciudad de los Ángeles. Tinseltown. ¡La Capital del Entretenimiento del Mundo!

Conocida por su clima cálido, sus palmeras, su extensa costa y Hollywood, además de ser cuna de algunas de las películas y canciones más icónicas. Pero, como en cualquier ciudad muy poblada, no siempre todo es glamur y puede haber un volumen importante de delitos. ¡Ahí es donde puedes ayudar!

Te han pedido que apoyes al Departamento de Policía de Los Ángeles (LAPD) analizando datos de criminalidad para identificar patrones de comportamiento delictivo. Planean usar tus conclusiones para asignar recursos de forma eficaz y combatir distintos delitos en diferentes zonas.

## Los datos

Te han proporcionado un único conjunto de datos. A continuación, tienes un resumen y una vista previa.

Es una versión modificada de los datos originales, que están disponibles públicamente en Los Angeles Open Data.

# crimes.csv

| Column     | Description              |
|------------|--------------------------|
| `'DR_NO'` | Division of Records Number: número de expediente oficial compuesto por un año de 2 dígitos, un ID de área y 5 dígitos. |
| `'Date Rptd'` | Fecha de denuncia - MM/DD/YYYY. |
| `'DATE OCC'` | Fecha de ocurrencia - MM/DD/YYYY. |
| `'TIME OCC'` | Hora en formato militar de 24 horas. |
| `'AREA NAME'` | Las 21 áreas geográficas o divisiones de patrulla también tienen un nombre que hace referencia a un lugar emblemático o a la comunidad a la que prestan servicio. Por ejemplo, la 77th Street Division está en la intersección de South Broadway y 77th Street y atiende barrios del sur de Los Ángeles. |
| `'Crm Cd Desc'` | Indica el delito cometido. |
| `'Vict Age'` | Edad de la víctima en años. |
| `'Vict Sex'` | Sexo de la víctima: `F`: mujer, `M`: hombre, `X`: desconocido. |
| `'Vict Descent'` | Ascendencia de la víctima:<ul><li>`A` - Other Asian (otro asiático)</li><li>`B` - Black (negro/a)</li><li>`C` - Chinese (chino/a)</li><li>`D` - Cambodian (camboyano/a)</li><li>`F` - Filipino (filipino/a)</li><li>`G` - Guamanian (guameño/a)</li><li>`H` - Hispanic/Latin/Mexican (hispano/latino/mexicano/a)</li><li>`I` - American Indian/Alaskan Native (indio/a americano/a/nativo/a de Alaska)</li><li>`J` - Japanese (japonés/japonesa)</li><li>`K` - Korean (coreano/a)</li><li>`L` - Laotian (laosiano/a)</li><li>`O` - Other (otro)</li><li>`P` - Pacific Islander (originario/a de islas del Pacífico)</li><li>`S` - Samoan (samoano/a)</li><li>`U` - Hawaiian (hawaiano/a)</li><li>`V` - Vietnamese (vietnamita)</li><li>`W` - White (blanco/a)</li><li>`X` - Unknown (desconocido)</li><li>`Z` - Asian Indian (indio/a asiático/a)</li> |
| `'Weapon Desc'` | Descripción del arma utilizada (si corresponde). |
| `'Status Desc'` | Estado del caso. |
| `'LOCATION'` | Dirección donde ocurrió el delito. |

In [100]:
# Re-run this cell
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
crimes = pd.read_csv("crimes.csv", dtype={"TIME OCC": str})
crimes.head()

,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA NAME,Crm Cd Desc,Vict Age,Vict Sex,Vict Descent,Weapon Desc,Status Desc,LOCATION
0,220314085,2022-07-22,2020-05-12,1110,Southwest,THEFT OF IDENTITY,27,F,B,NaN,Invest Cont,2500 S SYCAMORE AV
1,222013040,2022-08-06,2020-06-04,1620,Olympic,THEFT OF IDENTITY,60,M,H,NaN,Invest Cont,3300 SAN MARINO ST
2,220614831,2022-08-18,2020-08-17,1200,Hollywood,THEFT OF IDENTITY,28,M,H,NaN,Invest Cont,1900 TRANSIENT
3,231207725,2023-02-27,2020-01-27,0635,77th Street,THEFT OF IDENTITY,37,M,H,NaN,Invest Cont,6200 4TH AV
4,220213256,2022-07-14,2020-07-14,0900,Rampart,THEFT OF IDENTITY,79,M,B,NaN,Invest Cont,1200 W 7TH ST


In [101]:
crimes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 185715 entries, 0 to 185714
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   DR_NO         185715 non-null  int64 
 1   Date Rptd     185715 non-null  object
 2   DATE OCC      185715 non-null  object
 3   TIME OCC      185715 non-null  object
 4   AREA NAME     185715 non-null  object
 5   Crm Cd Desc   185715 non-null  object
 6   Vict Age      185715 non-null  int64 
 7   Vict Sex      185704 non-null  object
 8   Vict Descent  185705 non-null  object
 9   Weapon Desc   73502 non-null   object
 10  Status Desc   185715 non-null  object
 11  LOCATION      185715 non-null  object
dtypes: int64(2), object(10)
memory usage: 17.0+ MB


In [102]:
#Para responder a la primer pregunta, peak de crimenes por hora, comenzaré adaptando del formato militar al formato hora y luego haré un value_counts normalizado.


#Relleno la cadena con 0 hasta que haya 4 digitos
crimes['TIME OCC'] = crimes['TIME OCC'].astype(str).str.zfill(4)

#Genero una columna datetime con el tiempo convertido a formato hora
crimes['date_time'] = pd.to_datetime(crimes['TIME OCC'], format='%H%M')
#Extraigo la hora
crimes['TIME OCC'] = crimes['date_time'].dt.strftime('%H:%M')

crimes['TIME OCC'].value_counts(normalize=True).head(5)

12:00    0.034795
18:00    0.021490
20:00    0.020295
17:00    0.020214
00:01    0.019315
Name: TIME OCC, dtype: float64

In [103]:
#Genero la variable peal_crime_hour ahora
peak_crime_hour=crimes['TIME OCC'].value_counts().idxmax()
print(peak_crime_hour)

12:00


In [104]:
#Para comenzar a responder la pregunta 2 voy a generar un filtro de horas
crimes['hour'] = crimes['TIME OCC'].str.split(':').str[0].astype(int) #Hago esto pq las horas están en formato objeto
nocturno= crimes[(crimes['hour']>= 22) |(crimes['hour']< 4)]
peak_night_crime_location=nocturno['AREA NAME'].value_counts().idxmax()
print(peak_night_crime_location)

Central


In [105]:
#Delitos por grupo de Edad
#Comienzo generando una columna extra etaria
def edades(x):
    if x<=17:
        return '0-17'
    elif x>=18 and x<26:
        return '18-25'
    elif x>=26 and x<=34:
        return '26-34'
    elif x>=35 and x<=44:
        return '35-44'
    elif x>=45 and x<=54:
        return '45-54'
    elif x>=55 and x<=64:
        return '55-64'
    else:
        return '65+'

crimes['Range Age']=crimes['Vict Age'].apply(edades)


In [106]:
victim_ages=crimes['Range Age'].value_counts()
print(victim_ages)

26-34    47470
35-44    42157
45-54    28353
18-25    28291
55-64    20169
65+      14747
0-17      4528
Name: Range Age, dtype: int64


In [107]:
peak_crime_hour=crimes['hour'].value_counts().idxmax().astype(int)
peak_night_crime_location=nocturno['AREA NAME'].value_counts().idxmax()
victim_ages=crimes['Range Age'].value_counts()